In [4]:
import pandas as pd

In [5]:
df = pd.read_csv("PS_20174392719_1491204439457_log.csv")

In [6]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [7]:
print(df.shape)

(6362620, 11)


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            str    
 2   amount          float64
 3   nameOrig        str    
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        str    
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), str(3)
memory usage: 534.0 MB


In [9]:
df.isnull().sum()

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

### imbalanced dataset, dolandırıcılık işlemi %0.13 gibi bir oran. accuracy metriği yerine precision ve recall öne çıkıyor.

In [10]:
print(df['isFraud'].value_counts())

isFraud
0    6354407
1       8213
Name: count, dtype: int64


### fraud olanları seçip işlem türlerini belirledik. dolandırıcıların önce parayı kendi hesaplarına transfer edip sonra nakit olarak çektiklerini görebiliyoruz.

In [11]:
df[df['isFraud'] == 1]['type'].value_counts()

type
CASH_OUT    4116
TRANSFER    4097
Name: count, dtype: int64

### veriyi yalnızca cash_out ve transfer kalacak şekilde filtreledik.

In [12]:
df = df[df['type'].isin(['TRANSFER','CASH_OUT'])]

In [13]:
print(df.shape)

(2770409, 11)


### modelimizde kullanmayacağımız sütunları temizledik

In [14]:
df = df.drop(columns = ['nameOrig','nameDest','isFlaggedFraud'])

In [15]:
df.head()

,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud
2,1,TRANSFER,181.00,181.0,0.0,0.0,0.00,1
3,1,CASH_OUT,181.00,181.0,0.0,21182.0,0.00,1
15,1,CASH_OUT,229133.94,15325.0,0.0,5083.0,51513.44,0
19,1,TRANSFER,215310.30,705.0,0.0,22425.0,0.00,0
24,1,TRANSFER,311685.89,10835.0,0.0,6267.0,2719172.89,0


### str sütunları encode edelim. cash_out için 0, transfer için 1 kullanacağız.

In [16]:
df['type'] = df['type'].map({'CASH_OUT':0, 'TRANSFER':1})

In [17]:
df.info()

<class 'pandas.DataFrame'>
Index: 2770409 entries, 2 to 6362619
Data columns (total 8 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            int64  
 2   amount          float64
 3   oldbalanceOrg   float64
 4   newbalanceOrig  float64
 5   oldbalanceDest  float64
 6   newbalanceDest  float64
 7   isFraud         int64  
dtypes: float64(5), int64(3)
memory usage: 190.2 MB


### modelimizin eğitimi için veri setinin %80 ini testi içinse %20 sini kullanacağız. öncelikle girdileri ve çıktıları(isFraud) belirleyelim.

In [18]:
X = df.drop(columns=['isFraud'])
y = df['isFraud']

In [19]:
from sklearn.model_selection import train_test_split

In [20]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [21]:
X_train.shape

(2216327, 7)

### Dominance problemini çözmek için standardization yapmamız gerekiyor. 

In [22]:
from sklearn.preprocessing import StandardScaler

### StandardScaler(), verideki her bir sayı için Z-skoru hesaplaması yani bir veri değerinden ortalamanın çıkarılması ve sonucun standart sapmaya bölünmesi hesabını yapar. 
#### böylece sütunda 500k gibi büyük sayılar da olsa 10 gibi küçük sayılar da olsa hepsi 0 etrafında -3, +3 gibi kümelenir.

In [25]:
scaler = StandardScaler()

#### train setine fit kullandık çünkü modelin bu setteki ortalama, standart sapma gibi bilgileri öğrenmesini istiyoruz.
#### test setine yazmadık çünkü model, test veri seti hakkında hiçbir ön bilgiye sahip olmamalı (data leakage olmamalı)

In [26]:
X_train_scaled = scaler.fit_transform(X_train)

In [27]:
X_test_scaled = scaler.transform(X_test)

### Logistic Regression
#### Model, her bir attribute un önemine göre bir ağırlık atar. Sigmoid Fonksiyonu ile içeriye giren sayı 0 ile 1 arasında bir değere dönüştürülür. Çıkan sonuç 0.5 ten büyükse fraud olduğuna karar verir.

In [28]:
from sklearn.linear_model import LogisticRegression

In [42]:
model = LogisticRegression(max_iter=1000, random_state=42)

### model.fit çalışma mantığı
#### 1.Model ilk başta tamamen rastgele ağırlıklar atar.
#### 2. rastgele kurallarla X_train işlemleri tahmin etmeye çalışır.
#### 3. Gerçek cevaplara (y_train) bakar.Hatasının ne kadar büyük olduğunu hesaplar.
#### 4. Demek ki Tutar sütununun ağırlığını biraz daha artırmalıyım diyerek formülünü günceller.
#### 5. Bu süreci tekrar tekrar yapar. Kodda max_iter=1000 yazarak ona "Bu düzeltme işlemini en fazla 1000 kere yapabilirsin" dedik.

In [43]:
model.fit(X_train_scaled, y_train)
print("Model başarıyla eğitildi!")

Model başarıyla eğitildi!


### Modelimizi test edelim. verdiği cevapları karşılaştırıp Confusion matrix ve classification reportunu çıkaralım.

In [44]:
y_pred = model.predict(X_test_scaled)

In [45]:
from sklearn.metrics import confusion_matrix, classification_report

In [46]:
print("--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))

--- Confusion Matrix ---
[[552339     97]
 [   881    765]]


#### model 765 dolandırıcıyı yakaladı. 97 tane fraud olmayan işleme fraud, 881 adet fraud işleme ise normal dedi

In [47]:
print("\n--- Classification Report---")
print(classification_report(y_test, y_pred))


--- Classification Report---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    552436
           1       0.89      0.46      0.61      1646

    accuracy                           1.00    554082
   macro avg       0.94      0.73      0.80    554082
weighted avg       1.00      1.00      1.00    554082



#### model birine dolandırıcı dediyse %89 haklı çıktı (precision1 0.89). verideki toplam dolandırıcılarınsa yalnızca %46 sını yakalayabildik. (recall 0.46)

#### Dengesiz bir veri olduğu için model, masum müşterileri rahatsız etmemeye odaklandı ve %89 Kesinlik (Precision) verdi. Ancak dolandırıcıların sadece %46'sını yakaladı.

### predict_proba bize dolandırıcılık olasılığını verecek [normal olma ihtimali, fraud olma ihtimali]. 100 ile çarparak skora ulaşıyoruz.

In [50]:
fraud_probabilities = model.predict_proba(X_test_scaled)[:,1] * 100

In [51]:
tableau_df = X_test.copy()
tableau_df['Real_isFraud'] = y_test.values
tableau_df['Fraud_Score'] = fraud_probabilities.round(2)

In [52]:
tableau_df.head()

,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,Real_isFraud,Fraud_Score
1442460,140,0,285073.76,229352.64,0.0,0.00,285073.76,0,0.23
5847267,402,0,204657.38,5093.00,0.0,911883.62,1116541.00,0,0.00
2163940,184,0,27554.10,0.00,0.0,154247.81,181801.91,0,0.10
2689116,210,0,100950.61,0.00,0.0,3946646.29,4047596.90,0,0.01
1452693,140,0,42737.06,0.00,0.0,409890.88,364619.44,0,0.05


In [53]:
top_frauds = tableau_df.sort_values(by='Fraud_Score', ascending=False)
top_frauds.head(10)

,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,Real_isFraud,Fraud_Score
6019701,456,0,4017972.88,4017972.88,0.0,0.00,4017972.88,1,100.0
6045012,492,1,2985840.40,2985840.40,0.0,0.00,0.00,1,100.0
6272937,625,0,2654302.46,2654302.46,0.0,1429770.91,4084073.37,1,100.0
6361142,717,1,2624422.62,2624422.62,0.0,0.00,0.00,1,100.0
3247296,250,0,10000000.00,10000000.00,0.0,1640761.93,11640761.93,1,100.0
6296769,680,0,10000000.00,10000000.00,0.0,0.00,10000000.00,1,100.0
3611017,272,1,5141769.94,5141769.94,0.0,0.00,0.00,1,100.0
5841055,402,1,4257337.62,4257337.62,0.0,0.00,0.00,1,100.0
6064044,505,0,2251225.38,2251225.38,0.0,23722.50,2274947.89,1,100.0
6318817,687,0,3009125.59,3009125.59,0.0,46901.34,3056026.93,1,100.0


In [54]:
tableau_df.to_csv('Fraud_Scores.csv', index=False)
print("Data has exported successfully")

Data has exported successfully
